# Quark/gluon jet samples (Parquet)

Generates truth-labeled **quark** and **gluon** jet samples from Pythia8 QCD
dijet events, plus an **inclusive** (unfiltered) jet sample on the same
schema for verification. Constituents are written out per jet, in Parquet,
for a follow-up quark/gluon-tagging notebook (XGBoost, done separately with
a student).

**Truth labeling.** A jet is called quark/gluon if it lies within
`MATCH_R` of an outgoing parton of Pythia8's hardest 2→2 subprocess
(status code $\pm23$), matched by the parton's PDG id. This is the standard
*parton-based* truth definition used in the quark/gluon-tagging literature
(the same recipe behind, e.g., the public energyflow quark/gluon dataset).
It's an idealized label, not a perfect one — hadronization and parton-shower
radiation mean a jet's constituents don't come exclusively from the matched
parton, and jets with no nearby hard parton are left `'unmatched'`.

**Why also write `inclusive_jets.parquet`?** It's every jet that passes
`pt_min` and `ETA_MAX`, on the *same* columns as the pure samples (including the `flavor`
column). `quark_jets.parquet` / `gluon_jets.parquet` are literally
`inclusive_jets.parquet` filtered on `flavor` — so the inclusive file is a
built-in consistency check (do the pure samples add up to the right subset?)
and a natural "unbiased mix" to evaluate a trained tagger against.

**Kinematic note for later:** gluon jets and quark jets don't share the same
$p_T$ spectrum out of a single `HardQCD:all` run (gluons dominate at low
$p_T$/low-$x$) — if the downstream tagger should learn shape rather than
kinematics, that's a reweighting step for the modeling notebook, not this one.

<!-- teaching-note:physics-vocabulary -->

## Vocabulary: from a collision to a jet

Protons are made of **partons**—quarks and gluons. In a high-energy proton collision, one
parton from each proton can undergo a short-distance **hard scattering**. Quarks and gluons
carry color charge and cannot be observed alone. They radiate more quarks and gluons in a
**parton shower**, then form color-neutral particles through **hadronization**. A jet
algorithm gathers the resulting nearby particles into a jet.

The “quark jet” or “gluon jet” label therefore refers to the simulated hard parton associated
with a jet, not to a directly observed quark or gluon. Real detector data do not contain this
truth label, which is one reason simulation assumptions must be stated clearly.


In [ ]:
# heppyyier.load() is a no-op if packages were already loaded via
# `module load` or the heppyyier Jupyter kernel. Safe to leave in place.
import heppyyier
heppyyier.load('pythia8')
heppyyier.load('fastjet')

In [ ]:
import cppyy
import pythia8
import fastjet
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import pyarrow  # noqa: F401 -- pandas' parquet engine
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)

# Sanity check: cppyy location
print(f"cppyy from: {cppyy.__file__}")

PseudoJetVec = cppyy.gbl.std.vector[fastjet.PseudoJet]

def wrap_phi(phi):
    """Map any phi convention onto the standard (-pi, pi] range.

    FastJet's PseudoJet.phi() returns [0, 2*pi); Pythia8's Particle.phi()
    already returns (-pi, pi]. Route everything through this so distances
    and stored values are on a consistent convention.
    """
    return (phi + np.pi) % (2 * np.pi) - np.pi

def delta_r(eta1, phi1, eta2, phi2):
    deta = eta1 - eta2
    dphi = wrap_phi(phi1 - phi2)
    return float(np.hypot(deta, dphi))

## Configure Pythia8

<!-- teaching-note:monte-carlo -->

### What Pythia is simulating

Pythia is a **Monte Carlo event generator**. Monte Carlo means that it uses random sampling
from physics probability distributions to create possible collision events. One simulated
event is not a prediction by itself; distributions over many events are the prediction.
The random seed makes a run reproducible: the same seed and settings produce the same random
sequence. The hard-QCD setting requests strong-interaction scattering processes.


In [ ]:
import os
PYTHIA_SEED = 7  # edit for another reproducible random sequence
pythia = pythia8.Pythia()
pythia.readString('Beams:eCM = 13000.')
pythia.readString('HardQCD:all = on')
pythia.readString('PhaseSpace:pTHatMin = 20.')
pythia.readString('Random:setSeed = on')
pythia.readString(f'Random:seed = {PYTHIA_SEED}')
pythia.readString('Next:numberShowEvent = 0')
pythia.readString('Print:quiet = on')
pythia.init()

## Jet definition and run parameters

<!-- teaching-note:coordinates-and-jets -->

### Coordinates, acceptance, and the jet radius

The beam defines the longitudinal direction. Transverse momentum $p_T$ is perpendicular to
that beam. Pseudorapidity $\eta$ is a direction coordinate: $\eta=0$ is perpendicular to the
beam and large $|\eta|$ points closer to it. The requirement $|\eta|<2$ is an **acceptance
cut** that keeps jets in a central region where a collider detector can usually measure them
well.

FastJet's anti-$k_t$ algorithm repeatedly combines particles according to a distance rule.
Its radius parameter $R$ controls the typical angular reach of a jet. Angular distance is
$\Delta R=\sqrt{(\Delta\eta)^2+(\Delta\phi)^2}$, where $\phi$ is the angle around the beam.
Truth matching chooses a one-to-one jet–parton assignment with small $\Delta R$; it does not
claim that every particle in the jet came only from that parton.


In [ ]:
R        = 0.4
pt_min   = 20.0     # GeV -- strict jet pT threshold
ETA_MAX  = 2.0      # strict jet |eta| acceptance
N_EVENTS = 20_000  # edit this number; runtime scales approximately linearly
MATCH_R  = R         # parton<->jet truth-matching radius

QUARK_IDS = {1, 2, 3, 4, 5}   # d, u, s, c, b
GLUON_ID  = 21

OUT_DIR = 'data'  # edit to write the generated files elsewhere

jet_def = fastjet.JetDefinition(fastjet.antikt_algorithm, R)

## Cluster + truth-match helpers

`cluster_event()` builds PseudoJets from the current Pythia event (tagging
each with `set_user_index` so constituents can be traced back to their PDG
id) and clusters them. `match_jets_to_partons()` ΔR-matches jets to the
outgoing hard-process partons (status $\pm23$) and returns a truth label per
matched jet index.

In [ ]:
def cluster_event():
    """Cluster the current Pythia event. Returns (cs, jets, part_pdgid)."""
    part_pdgid = []
    particles = PseudoJetVec()
    for i in range(pythia.event.size()):
        p = pythia.event[i]
        if p.isFinal() and p.isVisible():
            pj = fastjet.PseudoJet(p.px(), p.py(), p.pz(), p.e())
            pj.set_user_index(len(part_pdgid))
            particles.push_back(pj)
            part_pdgid.append(p.id())

    cs = fastjet.ClusterSequence(particles, jet_def)
    accepted_jets = [jet for jet in cs.inclusive_jets(pt_min)
                     if jet.pt() > pt_min and abs(jet.eta()) < ETA_MAX]
    jets = sorted(accepted_jets, key=lambda jet: jet.pt(), reverse=True)
    return cs, jets, part_pdgid


def match_jets_to_partons(jets, event, match_r=MATCH_R):
    """Truth-match jets to the outgoing partons of the hardest subprocess.

    Returns {jet_index: (flavor, pdgid, delta_r)} for jets matched within
    `match_r` of a quark or gluon status-23 parton. The global one-to-one
    assignment first maximizes the number of matches, then minimizes total
    delta_r. Exact ties use event-record parton index and jet index. Jets
    with no match are omitted (caller treats them as 'unmatched').
    """
    partons = []
    for parton_index in range(event.size()):
        parton = event[parton_index]
        pid = parton.id()
        if abs(parton.status()) == 23 and (abs(pid) in QUARK_IDS or pid == GLUON_ID):
            partons.append((parton_index, parton))

    candidates = {}
    for parton_index, parton in partons:
        matches = []
        for jet_index, jet in enumerate(jets):
            dr = delta_r(jet.eta(), wrap_phi(jet.phi()), parton.eta(), parton.phi())
            if dr < match_r:
                matches.append((jet_index, dr))
        candidates[parton_index] = matches

    best_key = None
    best_assignment = ()

    def search(parton_pos, used_jets, assignment, total_dr):
        nonlocal best_key, best_assignment
        if parton_pos == len(partons):
            signature = tuple((parton_index, jet_index)
                              for parton_index, jet_index, _ in assignment)
            key = (-len(assignment), total_dr, signature)
            if best_key is None or key < best_key:
                best_key = key
                best_assignment = tuple(assignment)
            return

        parton_index, _ = partons[parton_pos]
        search(parton_pos + 1, used_jets, assignment, total_dr)
        for jet_index, dr in candidates[parton_index]:
            if jet_index not in used_jets:
                used_jets.add(jet_index)
                assignment.append((parton_index, jet_index, dr))
                search(parton_pos + 1, used_jets, assignment, total_dr + dr)
                assignment.pop()
                used_jets.remove(jet_index)

    search(0, set(), [], 0.0)

    flavor_map = {}
    partons_by_index = dict(partons)
    for parton_index, jet_index, dr in best_assignment:
        pid = partons_by_index[parton_index].id()
        flavor_map[jet_index] = ('gluon' if pid == GLUON_ID else 'quark', pid, dr)
    return flavor_map

## Generate the sample

One mixed `HardQCD:all` run feeds all three outputs: every jet satisfying
`jet_pt > pt_min` and `abs(jet_eta) < ETA_MAX` becomes a row of `inclusive_df`;
`quark_df`/`gluon_df` are just the
truth-matched subsets of it.

In [ ]:
records = []
n_generated = 0

for event_id in tqdm(range(N_EVENTS), desc='Generating events', unit='event'):
    if not pythia.next():
        continue

    n_generated += 1
    cs, jets, part_pdgid = cluster_event()
    if not jets:
        continue

    flavor_map = match_jets_to_partons(jets, pythia.event)

    for j, jet in enumerate(jets):
        flavor, pdgid, dr = flavor_map.get(j, ('unmatched', 0, np.nan))
        constituents = cs.constituents(jet)

        records.append({
            'event_id': event_id,
            'jet_id': j,
            'flavor': flavor,
            'flavor_pdgid': pdgid,
            'matched_dr': dr,
            'jet_pt': jet.pt(),
            'jet_eta': jet.eta(),
            'jet_phi': wrap_phi(jet.phi()),
            'jet_mass': jet.m(),
            'jet_e': jet.e(),
            'n_constituents': len(constituents),
            'const_pt':  [c.pt() for c in constituents],
            'const_eta': [c.eta() for c in constituents],
            'const_phi': [wrap_phi(c.phi()) for c in constituents],
            'const_pid': [part_pdgid[c.user_index()] for c in constituents],
        })


inclusive_df = pd.DataFrame.from_records(records)
quark_df     = inclusive_df[inclusive_df['flavor'] == 'quark'].reset_index(drop=True)
gluon_df     = inclusive_df[inclusive_df['flavor'] == 'gluon'].reset_index(drop=True)

print(f'\nDone: {N_EVENTS} events -> {len(inclusive_df)} jets total')

In [ ]:
import ipywidgets as widgets
widgets.IntProgress(value=5, min=0, max=10)

## Sanity checks

<!-- teaching-note:quality-assurance -->

### Why sanity checks matter

A **sanity check** tests simple consequences that should hold before any machine learning.
Here we verify the acceptance, uniqueness of assignments, and agreement between inclusive
and flavor-filtered tables. Passing these checks does not prove the simulation is physically
perfect, but failing one reveals a definite bookkeeping or selection error.


In [ ]:
counts = inclusive_df['flavor'].value_counts()
print('Jets by flavor:')
print(counts.to_string())
print(f"\nquark : gluon ratio = {counts.get('quark', 0) / max(counts.get('gluon', 1), 1):.2f}")
print(f"unmatched fraction  = {counts.get('unmatched', 0) / len(inclusive_df):.1%}")
print(f"median match dR (matched jets) = {inclusive_df['matched_dr'].median():.3f}")
print(f"maximum |jet eta| = {inclusive_df['jet_eta'].abs().max():.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {'quark': 'steelblue', 'gluon': 'tomato', 'unmatched': 'lightgray'}

ax = axes[0]
bins = np.linspace(pt_min, 150, 40)
for flavor, color in colors.items():
    sel = inclusive_df.loc[inclusive_df['flavor'] == flavor, 'jet_pt']
    ax.hist(sel, bins=bins, histtype='step', linewidth=1.5, color=color,
            density=True, label=f'{flavor} (n={len(sel)})')
ax.set_xlabel(r'Jet $p_T$ [GeV]')
ax.set_ylabel('Jets / bin (normalized)')
ax.set_title(r'Jet $p_T$ by truth flavor')
ax.legend()

ax = axes[1]
max_n = int(inclusive_df['n_constituents'].max())
nbins = range(0, max_n + 2)
for flavor, color in colors.items():
    sel = inclusive_df.loc[inclusive_df['flavor'] == flavor, 'n_constituents']
    ax.hist(sel, bins=nbins, histtype='step', linewidth=1.5, color=color,
            density=True, align='left', label=flavor)
ax.set_xlabel('Constituent multiplicity')
ax.set_ylabel('Jets / bin (normalized)')
ax.set_title('Constituent multiplicity by truth flavor')
ax.legend()

plt.tight_layout()
plt.savefig('demo_quark_gluon_samples.png', dpi=150)
plt.show()
print('Saved demo_quark_gluon_samples.png')
print("Gluon jets riding higher in multiplicity here is the expected signal --")
print("it's the single strongest quark/gluon discriminant and a good sanity check.")

## Write to Parquet

<!-- teaching-note:parquet-format -->

### Why use Parquet?

Parquet is a column-oriented binary data format. It stores column names and types, compresses
repeated structure efficiently, and lets later code read selected columns. Each row here is
one jet; the constituent columns contain lists because jets have different particle counts.
The manifest records settings and a SHA-256 fingerprint so later stages can identify the
exact sample they used.


In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

paths = {
    'quark':     os.path.join(OUT_DIR, 'quark_jets.parquet'),
    'gluon':     os.path.join(OUT_DIR, 'gluon_jets.parquet'),
    'inclusive': os.path.join(OUT_DIR, 'inclusive_jets.parquet'),
}

quark_df.to_parquet(paths['quark'], engine='pyarrow', index=False)
gluon_df.to_parquet(paths['gluon'], engine='pyarrow', index=False)
inclusive_df.to_parquet(paths['inclusive'], engine='pyarrow', index=False)

for name, path in paths.items():
    df = {'quark': quark_df, 'gluon': gluon_df, 'inclusive': inclusive_df}[name]
    size_kb = os.path.getsize(path) / 1024
    print(f'{name:<10} {len(df):>8} jets -> {path}  ({size_kb:.0f} KB)')

import hashlib, json
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'schema_version': 1, 'n_events_requested': N_EVENTS,
    'n_events_generated': n_generated, 'pythia_seed': PYTHIA_SEED,
    'sqrt_s_gev': 13000.0, 'pt_min_gev': pt_min, 'eta_max': ETA_MAX,
    'jet_R': R, 'match_R': MATCH_R,
    'jet_counts': {str(k): int(v) for k, v in inclusive_df['flavor'].value_counts().items()},
    'inclusive_sha256': file_sha256(paths['inclusive']),
}
manifest_path = os.path.join(OUT_DIR, 'quark_gluon_generation_manifest.json')
with open(manifest_path, 'w') as stream:
    json.dump(manifest, stream, indent=2, sort_keys=True)
print(f'Wrote generation manifest: {manifest_path}')


---
## For the tagging notebook

```python
import pandas as pd
quark_df     = pd.read_parquet('data/quark_jets.parquet')
gluon_df     = pd.read_parquet('data/gluon_jets.parquet')
inclusive_df = pd.read_parquet('data/inclusive_jets.parquet')
```

One row per jet. `const_pt` / `const_eta` / `const_phi` / `const_pid` are
list-valued columns, one entry per constituent (same length as
`n_constituents`) — that's the raw material for building per-jet features
(multiplicity, $p_T$ dispersion, girth/width, EEC-type variables, ...) for
the XGBoost model. `flavor` is the label (`'quark'` / `'gluon'` /
`'unmatched'`); drop `'unmatched'` rows for training on `quark_df`/`gluon_df`,
or use `inclusive_df` to sanity-check the trained model against an unbiased
mix. `matched_dr` and `flavor_pdgid` are there for QA, not features.